
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)
---

### Konteks / Skenario

Tim data engineering platform e-commerce meminta anda membangun **pipeline ETL produksi pertama** yang menggabungkan tiga sumber data sekaligus: transaksi (CSV), data produk (JSON), dan data ulasan pelanggan (CSV terpisah) — merepresentasikan kondisi nyata di mana data tersebar di berbagai sistem dengan format berbeda-beda.

### Menyiapkan Dataset

Jalankan cell berikut untuk menghasilkan **tiga sumber data** sekaligus.

In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


### Instruksi Pengerjaan

Buat `SparkSession`

---

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Tugas_6") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


**A. EXTRACT** *(bobot 15%)*

Baca ketiga sumber data (`tugas6_transaksi.csv`, `tugas6_produk.json`, `tugas6_ulasan.csv`) menjadi tiga Spark DataFrame terpisah. Tampilkan jumlah baris dan `printSchema()` masing-masing.

---

In [9]:
# EXTRACT — membaca tiga sumber data
df_transaksi = spark.read.csv(
    "tugas6_transaksi.csv",
    header=True,
    inferSchema=True
)

df_produk = spark.read.json(
    "tugas6_produk.json"
)

df_ulasan = spark.read.csv(
    "tugas6_ulasan.csv",
    header=True,
    inferSchema=True
)

print("Data transaksi:", df_transaksi.count(), "baris")
df_transaksi.printSchema()

print("Data produk:", df_produk.count(), "baris")
df_produk.printSchema()

print("Data ulasan:", df_ulasan.count(), "baris")
df_ulasan.printSchema()

Data transaksi: 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Data produk: 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Data ulasan: 3500 baris
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



**B. TRANSFORM — Penggabungan** *(bobot 25%)*

Gabungkan ketiga DataFrame menjadi satu (`order_id` sebagai kunci ke ulasan, `product_id` sebagai kunci ke produk). Gunakan **`salah satu join yang tepat`** dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan **`salah satu join yang tepat`** dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid). Tambahkan kolom `total_pendapatan` (`unit_terjual x harga`).

---

In [8]:
# TRANSFORM — menggabungkan ketiga DataFrame
df_etl = df_transaksi.join(
    df_ulasan,
    on="order_id",
    how="left"
)

df_etl = df_etl.join(
    df_produk,
    on="product_id",
    how="inner"
)

# Menambahkan kolom total_pendapatan
df_etl = df_etl.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga")
)

df_etl.show(5)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
o

**C. TRANSFORM — Penanganan Data Kosong & Pengayaan** *(bobot 20%)*

- Transaksi tanpa ulasan akan memiliki `rating` bernilai kosong (`null`) setelah `salah satu join yang tepat` — isi nilai kosong tersebut dengan angka **0** menggunakan `salah satu function`, sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".
- Tambahkan kolom `ada_ulasan` bernilai `True`/`False` (tidak boleh diisi manual satu satu)`, **sebelum** langkah `na.fill()` di atas).

---

In [10]:
# Menambahkan kolom ada_ulasan sebelum mengisi nilai kosong
df_etl = df_etl.withColumn(
    "ada_ulasan",
    col("rating").isNotNull()
)

# Mengisi rating yang kosong dengan 0
df_etl = df_etl.na.fill({"rating": 0})

df_etl.show(5)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|ada_ulasan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|      true|
|         8|     TX1|           6|2026-10-25 00:00:00|     0|250000|     Fashion|   Produk-8|         1500000|     false|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|      true|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|      true|
|        19|     TX4|           6|2026-10-07 00:00:00|     0| 75000|Rumah Tangga|  Produk-19|          450000|     false|
+----------+--------+---

**D. LOAD** *(bobot 25%)*

Simpan hasil akhir ke HDFS dalam format **Parquet**, dipartisi berdasarkan `kategori`, ke path `/user/[username]/tugas6/hasil_etl`. Verifikasi dengan `hdfs dfs -ls -R`, lalu baca kembali dan tampilkan `count()`-nya sebagai bukti data tersimpan utuh.

---

In [14]:
!hdfs dfs -mkdir -p /user/jauharaputri/tugas6

df_etl.write \
    .mode("overwrite") \
    .partitionBy("kategori") \
    .parquet(
        "hdfs://localhost:9000/user/jauharaputri/tugas6/hasil_etl"
    )

print("Data berhasil disimpan ke HDFS.")

!hdfs dfs -ls -R /user/jauharaputri/tugas6/hasil_etl

df_final = spark.read.parquet(
    "hdfs://localhost:9000/user/jauharaputri/tugas6/hasil_etl"
)

print("Jumlah baris hasil akhir:", df_final.count())

Data berhasil disimpan ke HDFS.
-rw-r--r--   3 jauharaputri supergroup          0 2026-09-30 16:39 /user/jauharaputri/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - jauharaputri supergroup          0 2026-09-30 16:39 /user/jauharaputri/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 jauharaputri supergroup      16363 2026-09-30 16:39 /user/jauharaputri/tugas6/hasil_etl/kategori=Elektronik/part-00000-6eed5db7-a91a-455f-8f49-4159bdbca17d.c000.snappy.parquet
drwxr-xr-x   - jauharaputri supergroup          0 2026-09-30 16:39 /user/jauharaputri/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 jauharaputri supergroup      11130 2026-09-30 16:39 /user/jauharaputri/tugas6/hasil_etl/kategori=Fashion/part-00000-6eed5db7-a91a-455f-8f49-4159bdbca17d.c000.snappy.parquet
drwxr-xr-x   - jauharaputri supergroup          0 2026-09-30 16:39 /user/jauharaputri/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 jauharaputri supergroup      10388 2026-09-30 16:39 /user/jauharaputri/tugas6/hasil_etl/kategori=

Jumlah baris hasil akhir: 5000


**E. Insight Akhir** *(bobot 15%)*

Dari data hasil ETL, tampilkan (menggunakan DataFrame API **atau** Spark SQL, bebas memilih): kategori produk mana yang memiliki **persentase transaksi dengan ulasan** (`ada_ulasan = True`) **paling rendah**? Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?

---

In [17]:
from pyspark.sql.functions import when

ringkasan_ulasan = df_final.groupBy("kategori").agg(
    count("*").alias("total_transaksi"),
    spark_sum(
        when(col("ada_ulasan") == True, 1).otherwise(0)
    ).alias("jumlah_dengan_ulasan")
)

ringkasan_ulasan = ringkasan_ulasan.withColumn(
    "persentase_ulasan",
    col("jumlah_dengan_ulasan") / col("total_transaksi") * 100
)

ringkasan_ulasan.orderBy("persentase_ulasan").show()

+------------+---------------+--------------------+-----------------+
|    kategori|total_transaksi|jumlah_dengan_ulasan|persentase_ulasan|
+------------+---------------+--------------------+-----------------+
|     Makanan|            536|                 369|68.84328358208955|
|   Kesehatan|            961|                 662|68.88657648283039|
|     Fashion|           1035|                 726|70.14492753623188|
|Rumah Tangga|            815|                 575| 70.5521472392638|
|  Elektronik|           1653|                1168|  70.659407138536|
+------------+---------------+--------------------+-----------------+



Kategori dengan persentase transaksi dengan ulasan paling rendah menunjukkan bahwa pelanggan pada kategori tersebut lebih sedikit memberikan ulasan. Informasi ini penting bagi tim marketing karena dapat menjadi bahan evaluasi untuk meningkatkan keterlibatan pelanggan, misalnya dengan mendorong pelanggan memberikan ulasan setelah melakukan pembelian.

---

## Menutup SparkSession

Kebiasaan baik: selalu tutup SparkSession di akhir notebook untuk clear up resources (memori & CPU) yang dipakainya.

In [18]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
